# DragonSR quickstart

`DragonSR` searches directed acyclic graphs of symbolic operators for a closed-form
expression of the data. This notebook runs it on two small problems with a reduced
budget (the paper's runs use 20,000 evaluations over 12 streams).

In [1]:
import logging, warnings
import numpy as np, pandas as pd
from dragon.search_algorithm.dragonsr import DragonSR

logging.disable(logging.WARNING); warnings.filterwarnings("ignore")

## A polynomial: Nguyen-4

$y = x^6 + x^5 + x^4 + x^3 + x^2 + x$ on $x \sim \mathcal{U}(-1, 1)$. One operator
family (`rational`) is enough here; each family is searched under both objectives,
so this run has two streams.

In [2]:
rng = np.random.default_rng(0)
x = rng.uniform(-1, 1, 2000)
X, y = pd.DataFrame({"x": x}), x**6 + x**5 + x**4 + x**3 + x**2 + x

model = DragonSR(n_iterations=240, families=["rational"], seed=1).fit(X, y)
print("expression:", model.formula_)
print("class:", model.winner_type_, "| stream:", model.stream_)

expression: (+1.0000*(x) -1.0000*(x**3)*(x * x**3)) / (-1.0000*(x) +1.0000)
class: rational | stream: rational|full


The reported expression is a ratio that reduces to the law:
$(x - x^7)/(1 - x) = x + x^2 + \dots + x^6$. `streams_` holds what every stream
reported, and `predict` evaluates the printed expression itself.

In [3]:
pd.DataFrame(model.streams_)[["stream", "loss", "winner_type", "actualT", "formula"]]

,stream,loss,winner_type,actualT,formula
0,rational|full,4.753309e-12,rational,122,(+1.0000*(x) -1.0000*(x**3)*(x * x**3)) / (-1....
1,rational|channel,1.908662e-02,channel,120,2.485070e+00 * (x**3 + x**2 + 0.212856 * x) + ...


In [4]:
xt = rng.uniform(-1, 1, 500)
print("R^2 on fresh data: %.6f" % model.score(pd.DataFrame({"x": xt}),
                                                 xt**6 + xt**5 + xt**4 + xt**3 + xt**2 + xt))

R^2 on fresh data: 1.000000


## A ratio of two inputs among several

An index of the NDVI type, $(b - a)/(b + a)$, with three distractor inputs. The
search sees all five inputs; XGBoost ranks them and variable augmentation may add
engineered features such as $(a+b)$.

In [5]:
rng = np.random.default_rng(1)
X2 = pd.DataFrame(rng.uniform(100, 3000, (2000, 5)), columns=["a", "b", "c", "d", "e"])
y2 = (X2.b - X2.a) / (X2.b + X2.a)

model2 = DragonSR(n_iterations=480, families=["rational", "log"], seed=5).fit(X2, y2)
print("expression:", model2.formula_)
print("class:", model2.winner_type_, "| stream:", model2.stream_, "| R^2: %.6f" % model2.score(X2, y2))

expression: (-31240242871.4590*(1 / (b + sign(b) * (abs(b) + 1e-30)**-3 + 1e-08)) +33036504423.3877*(1 / (a + sign(a) * (abs(a) + 1e-30)**-3 + 1e-08)) -3592523106.4050*(1 / (a + b + sign((a+b)) * (abs(a + b) + 1e-30)**-3 + 1e-08)) +0.0025) / (+31240242671.9063*(1 / (b + sign(b) * (abs(b) + 1e-30)**-3 + 1e-08)) +33036504214.1717*(1 / (a + sign(a) * (abs(a) + 1e-30)**-3 + 1e-08)) +1.0000)
class: rational | stream: rational|full | R^2: 1.000000


The printed expression carries guards (`1e-08`, `1e-30`) and redundant terms, as the search evaluated them, and can be the law in a form that does not resemble it. A small residual does not show that it is the law -- only reducing it does -- but it bounds how far the expression is from the law on these data:

In [6]:
err = np.abs(model2.predict(X2) - y2)
print("max |prediction - law| = %.2e, against values of order %.2f" % (err.max(), np.abs(y2).mean()))

max |prediction - law| = 8.85e-09, against values of order 0.35


## Settings

The defaults reproduce the configuration of the paper; the main knobs are
`n_iterations` (shared by all streams), `families` (operator families, six by
default), `objectives` (`("channel", "full")`), `subsample` (the paper scored each
candidate on 10% of the observations for noisy targets) and `seed`. The streams are
threads that share one random generator, so two runs with the same seed start from
the same state but are not identical bit for bit.